# 03. Directional Controls & Multi-Random Placebo Distribution (N=20)
Evaluates $+v_{\text{steer}}$, $-v_{\text{steer}}$, and 20 isotropic Gaussian random vectors with 20 published seeds.
Computes empirical randomization test $p$-value and hierarchical bootstrap CIs.

In [1]:
!pip install -q evaluate bert_score bitsandbytes accelerate transformers
import os, sys, json, time, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate
print('PyTorch:', torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.1 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128


In [2]:
possible_paths = [
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/thanhtranguyn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_root = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower() or 'vnese' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

if data_path is None:
    raise FileNotFoundError("Dataset file not found! Please check Kaggle input data sidebar.")

print('✅ Resolved Dataset Path:', data_path)
with open(data_path, 'r', encoding='utf-8') as f: full_dataset = json.load(f)
test_data = full_dataset[-500:]
train_pool = full_dataset[:-2205]
print('Total dataset size:', len(full_dataset), '| Test size:', len(test_data))
assert len(test_data) == 500, 'Test size must be 500'


✅ Resolved Dataset Path: /kaggle/input/datasets/thanhtranguyn/vnese-data/vietnamese_medical_halueval_15k_specialized.json
Total dataset size: 14700 | Test size: 500


In [3]:
model_id = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
bertscore = evaluate.load('bertscore')

pos_acts, neg_acts = [], []
for item in train_pool[:300]:
    q, pos_ans, neg_ans = item['question'], item.get('right_answer', item.get('positive_answer')), item['hallucinated_answer']
    t_pos = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}'
    t_neg = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}'
    with torch.no_grad():
        inp_p = tokenizer(t_pos, return_tensors='pt').to(model.device)
        pos_acts.append(model(inp_p.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
        inp_n = tokenizer(t_neg, return_tensors='pt').to(model.device)
        neg_acts.append(model(inp_n.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print('Vector v_steer ready! Shape:', v_steer.shape)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Vector v_steer ready! Shape: torch.Size([3584])


In [4]:
# Published 20 Seeds for Isotropic Gaussian Random Vectors
SEEDS = [42, 101, 202, 303, 404, 505, 606, 707, 808, 909, 111, 222, 333, 444, 555, 666, 777, 888, 999, 1234]
random_vectors = []
for seed in SEEDS:
    gen = torch.Generator().manual_seed(seed)
    v_rand = torch.randn(v_steer.shape, generator=gen)
    v_rand = v_rand / v_rand.norm(p=2)
    assert abs(float(v_rand.norm(p=2)) - 1.0) < 1e-5, 'Unit norm check failed'
    random_vectors.append(v_rand)
print(f'Generated {len(random_vectors)} unit-norm random vectors!')


Generated 20 unit-norm random vectors!


In [5]:
target_layer = model.model.layers[8]
results_list = []

def make_custom_hook(v_vec, alpha_0=18.0, K=16):
    step = 0
    def hook(module, inp, out):
        nonlocal step; step += 1
        alpha_t = alpha_0 * (1.0 - (step - 1) / K) if 1 <= step <= K else 0.0
        if alpha_t != 0.0:
            cur = out[0] if isinstance(out, tuple) else out
            v_curr = v_vec.to(device=cur.device, dtype=cur.dtype)
            mod = cur + alpha_t * v_curr
            return (mod,) + out[1:] if isinstance(out, tuple) else mod
        return out
    return hook

# Evaluate +v_steer and -v_steer
for cond, v_target in [('+v_steer', v_steer), ('-v_steer', -v_steer)]:
    print(f'Evaluating directional condition: {cond}...')
    for idx, item in enumerate(tqdm(test_data)):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inp = tokenizer(prompt, return_tensors='pt').to(model.device)
        p_len = inp.input_ids.shape[1]
        h_handle = target_layer.register_forward_hook(make_custom_hook(v_target))
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        h_handle.remove()
        gen_text = tokenizer.decode(out[0][p_len:], skip_special_tokens=True)
        results_list.append({
            'question_id': f'Q-{idx:03d}', 'condition': cond, 'seed': 0,
            'generated_text': gen_text,
            'gold_ref': item.get('right_answer', item.get('positive_answer')),
            'hal_ref': item['hallucinated_answer']
        })

df_res = pd.DataFrame(results_list)
os.makedirs('outputs', exist_ok=True)
df_res.to_csv('outputs/directional_controls_outputs.csv', index=False)
print('Saved outputs/directional_controls_outputs.csv!')


Evaluating directional condition: +v_steer...


100%|██████████| 500/500 [2:02:20<00:00, 14.68s/it]


Evaluating directional condition: -v_steer...


100%|██████████| 500/500 [1:51:28<00:00, 13.38s/it]

Saved outputs/directional_controls_outputs.csv!
